# Gridsecure - Machine Learning Models
**Project Title**: Gridsecure (An ML-based Electricity Theft Detection System)  
**Course**: Data Analytics & AI (JIIT Summer Internship Program)  
**Role**: Member 3 - Machine Learning Models Implementation & Comparison  

---
### Objectives
1. Implement three machine learning classifiers: **Logistic Regression**, **Decision Tree**, and **Random Forest**.
2. Perform stratified train-test split (80/20) to handle class imbalance (~8.5% theft rate).
3. Tune key hyperparameters and evaluate model performance using **Accuracy**, **Precision**, **Recall**, **F1-Score**, and **ROC-AUC**.
4. Generate a comprehensive **Model Comparison Table** and evaluation charts (Confusion Matrices, ROC Curves, Feature Importances).
5. Save the best trained model (`gridsecure_best_model.pkl`) for integration into the Gridsecure dashboard.


## 1. Import Libraries & Set Up Environment


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("ML Environment Ready!")


## 2. Load Cleaned Dataset


In [ ]:
# Upload 'electricity_theft_dataset_with_clusters_V2.csv' in Google Colab sidebar
data_path = 'electricity_theft_dataset_with_clusters_V2.csv'
df = pd.read_csv(data_path)

print("Dataset Shape:", df.shape)
print("Columns Overview:", df.columns.tolist()[:10])
df.head(3)


## 3. Feature Selection & Data Preprocessing


In [ ]:
# Drop non-predictive metadata & location columns
drop_cols = ["CONS_NO", "Locality", "City", "State"]
feature_df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Filter out raw daily date columns to focus on engineered features
date_cols = [c for c in feature_df.columns if c.startswith("2014") or c.startswith("2015") or c.startswith("2016")]
print(f"Filtering out {len(date_cols)} daily time-series columns for summary feature modeling...")
feature_df = feature_df.drop(columns=date_cols)

print("Engineered Features count:", feature_df.shape[1] - 1)

# Encode Categorical Features (e.g. Consumer_Type, Urban_Rural)
cat_cols = feature_df.select_dtypes(include=['object']).columns.tolist()
print("Categorical Columns to Encode:", cat_cols)

for col in cat_cols:
    le = LabelEncoder()
    feature_df[col] = le.fit_transform(feature_df[col].astype(str))

# Separate Features (X) and Target (y)
X = feature_df.drop(columns=["Theft_Flag"])
y = feature_df["Theft_Flag"]

# Handle infinite or NaN values in engineered ratio features
X = X.replace([np.inf, -np.inf], np.nan)
if X.isnull().sum().sum() > 0:
    X = X.fillna(X.median())
    print("Cleaned infinite and NaN values using median imputation.")

print(f"Features matrix shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts(normalize=True)*100}")


## 4. Stratified Train-Test Split & Feature Scaling


In [ ]:
# 80-20 Stratified Split to preserve theft class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")

# Standard Scaling for linear models (Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 5. Model Training & Hyperparameter Tuning


In [ ]:
results = {}
fitted_models = {}

# --- Model 1: Logistic Regression ---
print("Training Logistic Regression...")
log_reg = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)
y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

results["Logistic Regression"] = {
    "Accuracy": accuracy_score(y_test, y_pred_lr),
    "Precision": precision_score(y_test, y_pred_lr, zero_division=0),
    "Recall": recall_score(y_test, y_pred_lr, zero_division=0),
    "F1-Score": f1_score(y_test, y_pred_lr, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob_lr),
    "y_pred": y_pred_lr,
    "y_prob": y_prob_lr
}
fitted_models["Logistic Regression"] = log_reg

# --- Model 2: Decision Tree Classifier ---
print("Training Decision Tree Classifier...")
dt_clf = DecisionTreeClassifier(max_depth=7, min_samples_split=10, random_state=42, class_weight='balanced')
dt_clf.fit(X_train, y_train)

y_pred_dt = dt_clf.predict(X_test)
y_prob_dt = dt_clf.predict_proba(X_test)[:, 1]

results["Decision Tree"] = {
    "Accuracy": accuracy_score(y_test, y_pred_dt),
    "Precision": precision_score(y_test, y_pred_dt, zero_division=0),
    "Recall": recall_score(y_test, y_pred_dt, zero_division=0),
    "F1-Score": f1_score(y_test, y_pred_dt, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob_dt),
    "y_pred": y_pred_dt,
    "y_prob": y_prob_dt
}
fitted_models["Decision Tree"] = dt_clf

# --- Model 3: Random Forest Classifier ---
print("Training Random Forest Classifier...")
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, class_weight='balanced', n_jobs=-1)
rf_clf.fit(X_train, y_train)

y_pred_rf = rf_clf.predict(X_test)
y_prob_rf = rf_clf.predict_proba(X_test)[:, 1]

results["Random Forest"] = {
    "Accuracy": accuracy_score(y_test, y_pred_rf),
    "Precision": precision_score(y_test, y_pred_rf, zero_division=0),
    "Recall": recall_score(y_test, y_pred_rf, zero_division=0),
    "F1-Score": f1_score(y_test, y_pred_rf, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob_rf),
    "y_pred": y_pred_rf,
    "y_prob": y_prob_rf
}
fitted_models["Random Forest"] = rf_clf

print("\nModel Training Complete!")


## 6. Model Comparison Table


In [ ]:
comparison_df = pd.DataFrame(results).T[["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]]
print("================ MODEL COMPARISON TABLE ================")
display(comparison_df.round(4))

# Save comparison table
comparison_df.to_csv("Model_Comparison_Table.csv")
print("Saved 'Model_Comparison_Table.csv'")


## 7. Performance Visualizations


In [ ]:
# 1. Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx], cbar=False)
    axes[idx].set_title(f"{name} Confusion Matrix")
    axes[idx].set_xlabel("Predicted Label")
    axes[idx].set_ylabel("Actual Label")
plt.tight_layout()
plt.show()

# 2. ROC Curves Comparison
plt.figure(figsize=(8, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res["y_prob"])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {res['ROC-AUC']:.4f})", linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label="Random Baseline (AUC = 0.50)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves Comparison - Gridsecure Theft Detection")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

# 3. Random Forest Feature Importance
importances = rf_clf.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=feat_imp.values, y=feat_imp.index, hue=feat_imp.index, legend=False, palette="viridis")
plt.title("Top 15 Features Driving Electricity Theft Detection (Random Forest)")
plt.xlabel("Feature Importance Score")
plt.ylabel("Feature")
plt.show()


## 8. Export Best Model for Dashboard Integration


In [ ]:
best_model_name = comparison_df["F1-Score"].idxmax()
print(f"Best Performing Model (Highest F1-Score): {best_model_name}")

best_model = fitted_models[best_model_name]
joblib.dump(best_model, "gridsecure_best_model.pkl")
print("Saved best model to 'gridsecure_best_model.pkl' successfully!")
